In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle
import boto3
import datetime as dt

try:
    import snowflake.connector
except:
    ! pip install snowflake-connector-python
    import snowflake.connector

from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives.asymmetric import rsa, dsa
from cryptography.hazmat.primitives import serialization

try:
    import optbinning
except:
    ! pip install optbinning

try:
    import catboost
except:
    ! pip install catboost

In [ ]:
dtm_now = dt.datetime.now()
print(f'Latest run date: {dtm_now}')

#### Functions

In [ ]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # init client
    cls_client = boto3.client(
        's3',
    )
    # download file
    cls_client.download_file(
        str_project, 
        str_bucket_path, 
        str_local_path,
    )

#### Constants

In [ ]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

# output
str_dirname_output = './output'

#### Make output dir

In [ ]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [ ]:
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/01_data_collection/{str_filename}'
df = pd.read_parquet(str_uri)
# show
df

#### Mask negatives

In [ ]:
list_cols = [
    
]

#### Get the request month

In [ ]:
df['request_month'] = df['request_datetime'].apply(
    lambda x: str(x)[:7],
)
# show
df

#### Number of rows

In [ ]:
int_nrows = df.shape[0]
print(f'Number of Rows: {int_nrows}')

#### Structure

#### BK

In [ ]:
list_cols = [
    'ENG-bk',
]
for col in tqdm(list_cols):
    df[col] = df[col].mask(df[col] <0, np.nan)
dict_agg = {col: 'mean' for col in list_cols}
df_tmp = df.groupby(by='request_month', as_index=False).agg(dict_agg)

# plot
fig, ax = plt.subplots(figsize=(9,5))
ax.set_title(f'Mean BK Over Time (N = {int_nrows})')
ax.set_ylabel('Mean')
for col in tqdm(list_cols):
    ax.plot(df_tmp['request_month'], df_tmp[col], label=col)
ax.set_xticklabels(df_tmp['request_month'], rotation=90)
#ax.legend(loc='upper left', bbox_to_anchor=(1,1))
plt.tight_layout()
plt.show()

#### Income

In [ ]:
list_cols = [
    'fltgrossmonthly__income_sum',
]
for col in tqdm(list_cols):
    df[col] = df[col].mask(df[col] <0, np.nan)
dict_agg = {col: 'mean' for col in list_cols}
df_tmp = df.groupby(by='request_month', as_index=False).agg(dict_agg)

# plot
fig, ax = plt.subplots(figsize=(9,5))
ax.set_title(f'Mean Income Over Time (N = {int_nrows})')
ax.set_ylabel('Mean')
for col in tqdm(list_cols):
    ax.plot(df_tmp['request_month'], df_tmp[col], label=col)
ax.set_xticklabels(df_tmp['request_month'], rotation=90)
#ax.legend(loc='upper left', bbox_to_anchor=(1,1))
plt.tight_layout()
plt.show()

#### LTV

In [ ]:
list_cols = [
    'ENG-loan_to_value',
]
for col in tqdm(list_cols):
    df[col] = df[col].mask(df[col] <0, np.nan)
dict_agg = {col: 'mean' for col in list_cols}
df_tmp = df.groupby(by='request_month', as_index=False).agg(dict_agg)

# plot
fig, ax = plt.subplots(figsize=(9,5))
ax.set_title(f'Mean LTV Over Time (N = {int_nrows})')
ax.set_ylabel('Mean')
for col in tqdm(list_cols):
    ax.plot(df_tmp['request_month'], df_tmp[col], label=col)
ax.set_xticklabels(df_tmp['request_month'], rotation=90)
#ax.legend(loc='upper left', bbox_to_anchor=(1,1))
plt.tight_layout()
plt.show()

#### Miles

In [ ]:
list_cols = [
    'miles_odometer__app',
]
for col in tqdm(list_cols):
    df[col] = df[col].mask(df[col] <0, np.nan)
dict_agg = {col: 'mean' for col in list_cols}
df_tmp = df.groupby(by='request_month', as_index=False).agg(dict_agg)

# plot
fig, ax = plt.subplots(figsize=(9,5))
ax.set_title(f'Mean Vehicle Mileage Over Time (N = {int_nrows})')
ax.set_ylabel('Mean')
for col in tqdm(list_cols):
    ax.plot(df_tmp['request_month'], df_tmp[col], label=col)
ax.set_xticklabels(df_tmp['request_month'], rotation=90)
#ax.legend(loc='upper left', bbox_to_anchor=(1,1))
plt.tight_layout()
plt.show()

#### Payment history

In [ ]:
list_cols = [
    'flt_wtd_avg_open__tu_pmthx',
    'flt_wtd_avg_closed__tu_pmthx',
    'ENG-wtd_avg',
]
dict_agg = {col: 'mean' for col in list_cols}
df_tmp = df.groupby(by='request_month', as_index=False).agg(dict_agg)

fig, ax = plt.subplots(figsize=(9,5))
ax.set_title(f'Mean Payment History Over Time (N = {int_nrows})')
ax.set_ylabel('Mean')
for col in tqdm(list_cols):
    ax.plot(df_tmp['request_month'], df_tmp[col], label=col)
ax.set_xticklabels(df_tmp['request_month'], rotation=90)
ax.legend()
plt.tight_layout()
plt.show()

#### Credit builder tags

In [ ]:
list_cols_ignore = [
    'subjectage__ln',
    'has_inst_tag',
]
list_cols = [col for col in df.columns if 'tag' in col]
list_cols = [col for col in list_cols if col not in list_cols_ignore]

# get proportion
list_dict_row = []
for col in tqdm(list_cols):
    flt_mn = df[col].mean()
    dict_row = {
        'tag': col,
        'prop': flt_mn,
    }
    list_dict_row.append(dict_row)
df_tmp = pd.DataFrame(list_dict_row)
df_tmp.sort_values(by='prop', ascending=False, inplace=True)

# plot
fig, ax = plt.subplots(figsize=(15,5))
ax.set_title(f'Proportion by Credit Builder Tag (N = {int_nrows})')
ax.set_ylabel('Proportion')
bars = ax.bar(df_tmp['tag'], df_tmp['prop'])
ax.bar_label(bars, fmt='%.4f')
ax.set_xticklabels(df_tmp['tag'], rotation=90)
plt.show()

In [ ]:
list_cols_ignore = [
    'has_inst_tag',
    'CHIME-STRIDE_tag',
    'subjectage__ln',
]
list_cols = [col for col in df.columns if 'tag' in col]
list_cols = [col for col in list_cols if col not in list_cols_ignore]
dict_agg = {col: 'mean' for col in list_cols}
df_tmp = df.groupby(by='request_month', as_index=False).agg(dict_agg)

fig, ax = plt.subplots(figsize=(9,5))
ax.set_title(f'Proportion of Credit Builders Over Time (N = {int_nrows})')
ax.set_ylabel('Proportion')
for col in tqdm(list_cols):
    ax.plot(df_tmp['request_month'], df_tmp[col], label=col.split('_tag')[0])
ax.set_xticklabels(df_tmp['request_month'], rotation=90)
ax.legend(loc='upper left', bbox_to_anchor=(1,1))
plt.tight_layout()
plt.show()

#### Inquiries

In [ ]:
list_cols = [
    'g232s__tu',
    'g960s__tu',
]
for col in tqdm(list_cols):
    df[col] = df[col].mask(df[col] <0, np.nan)
dict_agg = {col: 'mean' for col in list_cols}
df_tmp = df.groupby(by='request_month', as_index=False).agg(dict_agg)

# plot
fig, ax = plt.subplots(figsize=(9,5))
ax.set_title(f'Mean Inquiries Over Time (N = {int_nrows})')
ax.set_ylabel('Mean')
for col in tqdm(list_cols):
    ax.plot(df_tmp['request_month'], df_tmp[col], label=col)
ax.set_xticklabels(df_tmp['request_month'], rotation=90)
ax.legend()
plt.tight_layout()
plt.show()

#### Repos

In [ ]:
list_cols = [
    'rp01s__tu',
    'au06s__tu',
]
for col in tqdm(list_cols):
    df[col] = df[col].mask(df[col] <0, np.nan)
dict_agg = {col: 'mean' for col in list_cols}
df_tmp = df.groupby(by='request_month', as_index=False).agg(dict_agg)

# plot
fig, ax = plt.subplots(figsize=(9,5))
ax.set_title(f'Mean Repossessions Over Time (N = {int_nrows})')
ax.set_ylabel('Mean')
for col in tqdm(list_cols):
    ax.plot(df_tmp['request_month'], df_tmp[col], label=col)
ax.set_xticklabels(df_tmp['request_month'], rotation=90)
ax.legend()
plt.tight_layout()
plt.show()

#### Open to buy

In [ ]:
list_cols = [
    'g201a__tu',
    'g202a__tu',
    'rev322__tu',
    'bkc322__tu',
]
for col in tqdm(list_cols):
    df[col] = df[col].mask(df[col] <0, np.nan)
dict_agg = {col: 'mean' for col in list_cols}
df_tmp = df.groupby(by='request_month', as_index=False).agg(dict_agg)

# plot
fig, ax = plt.subplots(figsize=(9,5))
ax.set_title(f'Mean Open to Buy Over Time (N = {int_nrows})')
ax.set_ylabel('Mean')
for col in tqdm(list_cols):
    ax.plot(df_tmp['request_month'], df_tmp[col], label=col)
ax.set_xticklabels(df_tmp['request_month'], rotation=90)
ax.legend()
plt.tight_layout()
plt.show()

#### Balance magnitude

In [ ]:
list_cols = [
    'balmag01__tu',
    'balmag02__tu',
]
for col in tqdm(list_cols):
    df[col] = df[col].mask(df[col] <0, np.nan)
dict_agg = {col: 'mean' for col in list_cols}
df_tmp = df.groupby(by='request_month', as_index=False).agg(dict_agg)

# plot
fig, ax = plt.subplots(figsize=(9,5))
ax.set_title(f'Mean Balance Magnitude Over Time (N = {int_nrows})')
ax.set_ylabel('Mean')
for col in tqdm(list_cols):
    ax.plot(df_tmp['request_month'], df_tmp[col], label=col)
ax.set_xticklabels(df_tmp['request_month'], rotation=90)
ax.legend()
plt.tight_layout()
plt.show()